# Chapter 9: Grouping and Aggregating

The programs from Chapter 9 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## groupby

`groupby_basics.py`

In [1]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")
orders["revenue"] = (
    orders["quantity"] * orders["unit_price"] * (1 - orders["discount"])
)

print(orders.groupby("region")["revenue"].sum().round(0))
print()
by_region = orders.groupby("region").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    customers=("customer_id", "nunique"),
)
print(by_region.round(1).sort_values("revenue", ascending=False))

region
East      98878.0
North    113274.0
South     86223.0
West      81328.0
Name: revenue, dtype: float64

        orders   revenue  avg_order  customers
region                                        
North     1768  113274.2       64.1        352
East      1560   98878.3       63.4        316
South     1300   86222.9       66.3        272
West      1205   81328.0       67.5        251


## Pivot Tables and Crosstabs

`pivot.py`

In [2]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
orders["quarter"] = orders["order_date"].dt.quarter

table = orders.pivot_table(
    index="category",
    columns="quarter",
    values="revenue",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
)
print((table / 1000).round(1))  # thousands of dollars
print()
print(
    pd.crosstab(
        orders["region"], orders["channel"], normalize="index"
    ).round(2)
)

quarter      1     2     3      4  Total
category                                
Bedding   20.5  21.0  25.6   43.8  110.9
Decor      7.1   7.5   9.5   15.6   39.7
Garden     7.6  23.6   6.8   12.6   50.5
Kitchen   16.2  14.5  21.2   34.4   86.3
Lighting  17.5  22.5  25.6   44.4  109.9
Total     68.9  89.0  88.7  150.8  397.4

channel   app  marketplace   web
region                          
East     0.28         0.20  0.51
North    0.31         0.20  0.49
South    0.30         0.21  0.50
West     0.32         0.20  0.48


## Share of Total With transform

`share_of_total.py`

In [3]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")
orders["revenue"] = orders["quantity"] * orders["unit_price"]

sales = orders.groupby(["region", "category"], as_index=False)[
    "revenue"
].sum()
# transform returns one value per row, aligned with the original rows
sales["share"] = sales["revenue"] / sales.groupby("region")[
    "revenue"
].transform("sum")
top = sales.sort_values("share", ascending=False).groupby("region").head(1)
print(top.round({"revenue": 0, "share": 3}).to_string(index=False))

region category  revenue  share
  West Lighting  24474.0  0.288
 North  Bedding  33780.0  0.285
 South  Bedding  25440.0  0.283
  East  Bedding  28445.0  0.274
